# Vertex AI Style Training Workflow

Goal: build an ML training workflow locally and document how it maps to Vertex AI training.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import joblib

OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.head()

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f'accuracy={accuracy:.4f}')
print(classification_report(y_test, predictions, target_names=iris.target_names))

In [ ]:
model_path = OUTPUT_DIR / 'iris_random_forest.joblib'
joblib.dump(model, model_path)
print(f'saved model to {model_path}')

## Vertex AI Mapping

- Notebook logic becomes `train.py`
- Dependencies move to `requirements.txt`
- Model artifact is uploaded to Cloud Storage
- Training job runs on Vertex AI Custom Training
- Model can be deployed to Vertex AI Endpoint or served on GKE/Cloud Run

In [ ]:
# from google.cloud import aiplatform
# aiplatform.init(project='YOUR_PROJECT_ID', location='us-central1', staging_bucket='gs://YOUR_BUCKET')
# job = aiplatform.CustomTrainingJob(
#     display_name='iris-rf-training',
#     script_path='train.py',
#     container_uri='us-docker.pkg.dev/vertex-ai/training/scikit-learn-cpu.1-3:latest',
#     requirements=['scikit-learn', 'joblib', 'pandas'],
# )
# model = job.run(replica_count=1, machine_type='n1-standard-4')